# Analyze example_experiment results

1. Scan `results/`
2. Load one candidate npz
3. Plots (will be added)

In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd

from COMIND_transformer.preprocessing import build_connectome, parse_data

# Open this notebook from example_experiment/ (or set EXP_DIR explicitly).
EXP_DIR = os.getcwd()
RESULTS_DIR = os.path.join(EXP_DIR, "results")

FINAL_RE = re.compile(r"result_(\d+)_")
HYPER_KEYS = (
    "n_subtypes", "lambda_f", "lambda_kappa", "lambda_cog",
    "lambda_scalar", "scalar_K_center", "lambda_jsd", "lambda_beta",
)
DISPLAY_COLS = [
    "candidate", "n_subtypes",
    "lambda_f", "lambda_kappa", "lambda_scalar", "scalar_K_center", "lambda_jsd",
    "lse_final", "val_lse", "bic", "bic_n_params", "n_obs", "path",
]

def row_from_npz(path: str) -> dict:
    d = np.load(path, allow_pickle=True)
    m = FINAL_RE.search(os.path.basename(path))
    cid = int(d["candidate"]) if "candidate" in d.files else (int(m.group(1)) if m else -1)
    row = {
        "candidate": cid,
        "path": path,
        "lse_final": float(d["lse_final"]) if "lse_final" in d.files else np.nan,
        "val_lse": float(d["val_lse"]) if "val_lse" in d.files else np.nan,
        "bic": float(d["bic"]) if "bic" in d.files else np.nan,
        "bic_n_params": int(d["bic_n_params"]) if "bic_n_params" in d.files else np.nan,
        "n_obs": int(d["n_obs"]) if "n_obs" in d.files else np.nan,
    }
    for k in HYPER_KEYS:
        if k in d.files:
            row[k] = int(d[k]) if k == "n_subtypes" else float(d[k])
    return row


def load_results_table(results_dir: str = RESULTS_DIR) -> pd.DataFrame:
    rows = []
    for path in glob.glob(os.path.join(results_dir, "result_*.npz")):
        rows.append(row_from_npz(path))
    if not rows:
        raise FileNotFoundError(f"No result_*.npz under {results_dir}")
    df = pd.DataFrame(rows)
    for c in DISPLAY_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[DISPLAY_COLS].sort_values(["bic", "lse_final"], na_position="last").reset_index(drop=True)


results_df = load_results_table()
print(f"Loaded {len(results_df)} candidates from {RESULTS_DIR}")
print(f"BIC range: [{results_df['bic'].min():.2f}, {results_df['bic'].max():.2f}]")
display(results_df.head(20))

In [ ]:
# Pick lowest-BIC by default; set CANDIDATE = <int> to pin one
CANDIDATE = int(results_df.loc[results_df["bic"].idxmin(), "candidate"])

best_row = results_df.loc[results_df["candidate"] == CANDIDATE].iloc[0]
result = np.load(best_row["path"], allow_pickle=True)

print(
    f"Candidate {CANDIDATE}: n_subtypes={int(best_row['n_subtypes'])}, "
    f"BIC={best_row['bic']:.4f}, LSE={best_row['lse_final']:.4f}, "
    f"val_LSE={best_row['val_lse']:.4f}"
)
print("path:", best_row["path"])
print("hypers:", {k: best_row[k] for k in HYPER_KEYS if k in best_row.index and pd.notna(best_row[k])})
if "experiment_name" in result.files:
    print("experiment:", result["experiment_name"])
if "csv_path" in result.files:
    print("csv:", result["csv_path"])
if "split_seed" in result.files:
    print(f"split_seed={int(result['split_seed'])}, test_size={float(result['test_size'])}")
print("npz keys:", sorted(result.files)[:20], "...")